# Boosting

In the previous lectures, we studied CARTs, bagging, and random forests. A single tree is flexible but unstable. Bagging and random forests reduce this instability by averaging many trees that are fit in parallel.

Boosting takes a different approach. Instead of fitting many trees independently and averaging them, boosting fits trees **sequentially**. 

> Each new tree is chosen to improve the current model, given where the previous model did poorly.

A generic boosting method constructs a sequence of score functions

$$
\hat s_0,\hat s_1,\dots,\hat s_M.
$$

Each score function in our sequence is going to focus on where the previous did poorly and try to compensate. At step $m$, we do this by fitting a new **weak learner** $\hat h_m$ (somehow) focused on where $\hat s_{m-1}$ did poorly, and then creating $\hat s_m$ as:

$$
\hat s_m(x)=\hat s_{m-1}(x)
+
\nu \hat h_m(x).
$$

for some **learning rate** $\nu \in (0,1]$. Consequently, after $M$ steps our boosted method will have the basic form of an additive score function. For regression this score is scalar:

$$
\hat s_M(x) = 
\hat s_0(x)
+
\sum_{m=1}^M \nu \hat h_m(x),
$$

where

* $\hat s_0$ is an initial score function
* $\hat h_m$ is the $m^{th}$ **weak learner** (usually a shallow tree)
* $M$ is the number of **boosting** iterations/steps
* $\nu \in (0,1]$ is the **learning rate** or **shrinkage factor**

The prediction function is then

$$
\hat f_M(x) = a(\hat s_M(x)),
$$

where $a$ is the action function. For regression, often $a(s)=s$.

For multiclass classification, the score function is vector-valued:

$$
\hat s_M(x) = 
\left(
\hat s_{M1}(x),\ldots,\hat s_{MK}(x)
\right),
$$

with one score for each class. Each of these individual scores functions are built up sequentially also just like the regression function. The action function is then

$$
a(\hat s_M(x))= 
\arg\max_{k \in {1,\ldots,K}} \hat s_{Mk}(x).
$$


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.datasets import make_moons, load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, log_loss

rng = np.random.default_rng(657677)

# Boosted regression trees with squared error

To make things concrete, let's first study the simplest boosting algorithm: boosted regression trees (for squared error loss). Assume we're solving a regression problem. Our generic boosting algorithm is: 

1. Fit $\hat s_0$.
2. For $m=1,\ldots,M$
    1. Determine where $\hat s_{m-1}$ did poorly,
    2. Fit $\hat h_m$ to do well where $\hat s_{m-1}$ didn't,
    3. Boost: $\hat s_m(x) = \hat s_{m-1}(x) + \nu \hat h_m(x)$.
3. Return $\hat s(x) = \hat s_M(x)$.

For our concrete boosted regression trees, we first need a base method, let's use the simple mean constant prediction: 

$$
\hat s_0(x)=\bar y.
$$

Now, at iteration $m$ we need to determine where $\hat s_{m-1}$ did poorly. Let's do this by calcualting the the current residuals of $s_{m-1}$ compared to the training data: 

$$
r_{nm}=y_n-\hat s_{m-1}(x_n).
$$

We're then going to try and boost our current approach to compensate. We'll do this by fitting a small regression tree $\hat h_m$ to these residuals. 

Then we boost using this **weak** regression tree: 

$$
\hat s_m(x)=
\hat s_{m-1}(x)
+
\nu \hat h_m(x).
$$

This is the basic boosted regression tree algorithm.


The learning rate $\nu$ controls how fast we boost. It is (very) akin to the learning rate in a gradient descent. We don't want to boost too quickly (over-fitting), we want a series of small steps. When $\nu=1$, each tree's fitted residual is added completely. When $\nu$ is small, each tree only makes a small correction.

Small learning rates usually require larger $M$, but they often improve test performance. The pair $(\nu,M)$ is therefore a regularization pair:

- smaller $\nu$ means slower learning
- larger $M$ means more additive components


## A small simulated example


In [ ]:

N = 180
X = rng.uniform(0, 1, size=(N, 1))

def true_regression_function(x):
    return np.sin(2 * np.pi * x) + 0.5 * np.sin(6 * np.pi * x)

y = true_regression_function(X[:, 0]) + rng.normal(0, 0.25, size=N)

x_grid = np.linspace(0, 1, 500).reshape(-1, 1)
y_true_grid = true_regression_function(x_grid[:, 0])

plt.figure(figsize=(8, 4))
plt.scatter(X[:, 0], y, alpha=0.6, label="observed")
plt.plot(x_grid[:, 0], y_true_grid, linewidth=2, label="true function")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Simulated regression data")
plt.legend()
plt.show()


## Implementing the boosting part ourselves

We will use `DecisionTreeRegressor` to fit each individual tree, but we will write the boosting loop directly.

This helps separate two ideas:

1. CART tells us how to fit one tree.
2. Boosting tells us how to combine many trees sequentially.


In [ ]:
class SimpleBoostedRegressionTrees:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=2, min_samples_leaf=5):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf

    def fit(self, X, y):
        self.init_ = np.mean(y)
        self.trees_ = []
        self.train_mse_ = []

        current_pred = np.repeat(self.init_, len(y))

        for m in range(self.n_estimators):
            residual = y - current_pred

            # fit tree to residuals
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                random_state=m
            )
            tree.fit(X, residual)

            # update *for training only*
            update = tree.predict(X)
            current_pred = current_pred + self.learning_rate * update

            self.trees_.append(tree)
            self.train_mse_.append(mean_squared_error(y, current_pred))

        return self

    def predict(self, X, n_estimators=None):
        if n_estimators is None:
            n_estimators = len(self.trees_)

        pred = np.repeat(self.init_, X.shape[0])
        for tree in self.trees_[:n_estimators]:
            pred = pred + self.learning_rate * tree.predict(X)

        return pred


In [ ]:
boost = SimpleBoostedRegressionTrees(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=8
)

boost.fit(X, y)

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(boost.train_mse_) + 1), boost.train_mse_)
plt.xlabel("Boosting iteration")
plt.ylabel("Training MSE")
plt.title("Training loss over boosting iterations")
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(X[:, 0], y, alpha=0.35, label="observed")
plt.plot(x_grid[:, 0], y_true_grid, linewidth=2, label="true function")

for m in [1, 5, 20, 200]:
    plt.plot(
        x_grid[:, 0],
        boost.predict(x_grid, n_estimators=m),
        linewidth=2,
        label=f"M = {m}"
    )

plt.xlabel("x")
plt.ylabel("prediction")
plt.title("Boosted regression tree predictions")
plt.legend()
plt.show()


## Effect of the learning rate

The learning rate controls how quickly the model moves. To see this, we can fit the same boosted tree procedure with different values of $\nu$.


In [ ]:
learning_rates = [1.0, 0.3, 0.1, 0.03]
histories = {}

for nu in learning_rates:
    model = SimpleBoostedRegressionTrees(
        n_estimators=200,
        learning_rate=nu,
        max_depth=2,
        min_samples_leaf=8
    )
    model.fit(X, y)
    histories[nu] = model.train_mse_

plt.figure(figsize=(8, 5))
for nu, hist in histories.items():
    plt.plot(np.arange(1, len(hist) + 1), hist, label=f"nu = {nu}")

plt.xlabel("Boosting iteration")
plt.ylabel("Training MSE")
plt.title("Learning rate and training loss")
plt.legend()
plt.show()


Large learning rates reduce training loss quickly. Small learning rates move more slowly, but the slower path is often better for test error.

In practice, boosted trees are usually tuned using a validation set or cross-validation. Common tuning parameters include:

- $M$: number of trees
- $\nu$: learning rate
- tree depth or number of leaves
- minimum leaf size

# General gradient boosting

The squared error algorithm is a special case of a broader idea.

For squared error, we fit residuals:

$$
r_n^{(m)} = y_n-\hat s_{m-1}(x_n).
$$

But residuals are not the right object for every loss. For example, in classification, the outcome is categorical, so the idea of fitting ordinary residuals doesn't make a lot of sense. 

Gradient boosting generalizes the residual-fitting idea by replacing residuals with **pseudo-residuals**.

For a loss function $\ell$, the empirical risk is

$$
\hat R(s)=
\frac{1}{N}\sum_{n=1}^N
\ell(y_n,s(x_n)).
$$

In parametric models, if $s=s_w$ for some parameter vector $w$, then we optimize $\hat R(s)=\hat R(w)$ over $w$. The most basic way we have done this is with gradient descent:

$$
w_m = w_{m-1} - \nu \nabla_w \hat R(w_{m-1}),
$$

where $\nu$ is the learning rate and $\nabla_w \hat R(w_{m-1})$ is the gradient of $\hat R$ with respect to $w$, evaluated at the current parameter value.

**Boosting is like gradient descent, but in function space.**

In boosting, we think of minimizing over functions $s$ directly. The current estimate is a function $\hat s_{m-1}$, and we update it by adding another function $\hat h_m$:

$$
\hat s_m(x)=
\hat s_{m-1}(x)
+
\nu \hat h_m(x).
$$

This is analogous to gradient descent, with $\hat h_m$ playing the role of a negative gradient direction. The difference is that instead of taking steps in a finite-dimensional parameter space, we take steps in a function space. This is why gradient boosting is often described as **gradient descent in function space**.

The idea is to start with an initial guess $\hat s_0$, then repeatedly approximate the negative functional derivative of $\hat R(s)$ with respect to $s$, evaluated at the current function $s=\hat s_{m-1}$. The weak learner $\hat h_m$ is fit to approximate this negative gradient direction (at the training points), and then we take a $\nu$-sized step in that direction.


## What is a functional gradient?

(For a longer discussion see: [https://mbernste.github.io/posts/functionals/](https://mbernste.github.io/posts/functionals/)).

In ordinary gradient descent, we minimize a function of parameters. For example, if the score function is $s_w(x)$, then the empirical risk can be written as a function of $w$:

$$
\hat R(w)=\frac{1}{N}\sum_{n=1}^N \ell(y_n,s_w(x_n)).
$$

The gradient tells us how $\hat R(w)$ changes if we make a small change to the parameter vector $w$. Then gradient descent moves $w$ in the direction that most decreases the risk.

In gradient boosting, we take a different point of view. Instead of thinking of $s$ as a function indexed by parameters $w$, we think of $s$ itself as the object we are changing. The empirical risk is now a function of a function:

$$
\hat R(s)=\frac{1}{N}\sum_{n=1}^N \ell(y_n,s(x_n)).
$$

An object that takes a function as input and returns a number is called a **functional**. Here, $\hat R$ takes the score function $s$ as input and returns the average loss on the training data.

So what does it mean to take a derivative with respect to a function?

The basic idea is the same as a directional derivative. In finite-dimensional calculus, if we have a function $R(w)$ and move from $w$ in the direction $v$, we study

$$
\left. \frac{d}{d\epsilon} R(w+\epsilon v) \right|_{\epsilon=0}.
$$

This tells us how quickly $R$ changes if we take a small step in the direction $v$. The gradient is the object whose inner product with $v$ gives this directional derivative:

$$
\left. \frac{d}{d\epsilon} R(w+\epsilon v) \right|_{\epsilon=0}= \nabla_w R(w)^\top v.
$$

For a functional, the input is not a vector $w$, but a function $s$. So instead of perturbing $w$ in the direction of a vector $v$, we perturb $s$ in the direction of another function $g$. For a small number $\epsilon$, define

$$
s_\epsilon(x)=s(x)+\epsilon g(x).
$$

This means that we are modifying the current score function $s(x)$ by adding a small amount of the function $g(x)$.

The directional derivative of the risk functional at $s$ in the direction $g$ is

$$
\left. \frac{d}{d\epsilon} \hat R(s+\epsilon g) \right|_{\epsilon=0}.
$$

The **functional gradient** is the object whose inner product with $g$ gives this directional derivative. Symbolically, it is the analogue of $\nabla_w R(w)$, but now the input being changed is the function $s$, not a parameter vector $w$. In our finite-sample setting, the risk only depends on the fitted values $s(x_1),\ldots,s(x_N)$, so the relevant inner product becomes a finite sum over the training data, like a typical inner product. If we perturb $s$ in the direction $g$, then

$$
\hat R(s+\epsilon g)=\frac{1}{N}\sum_{n=1}^N \ell(y_n,s(x_n)+\epsilon g(x_n)).
$$

Using the chain rule,

$$
\left. \frac{d}{d\epsilon} \hat R(s+\epsilon g) \right|_{\epsilon=0}=
\frac{1}{N}\sum_{n=1}^N \frac{\partial \ell(y_n,s(x_n))}{\partial s(x_n)}g(x_n).
$$

This has exactly the same structure as an inner product:

$$
\sum_n a_n b_n.
$$
where the $a_n$ are $\frac{\partial \ell(y_n,s(x_n))}{\partial s(x_n)}$ and the $b_n$ are $g(x_n)$. 

So the functional gradient, evaluated at the training points, is just the gradient of $\hat R$ with respect to these $N$ score values:

$$
\left(\frac{\partial \ell(y_1,s(x_1))}{\partial s(x_1)},\ldots,\frac{\partial \ell(y_N,s(x_N))}{\partial s(x_N)}\right).
$$

At each training point $x_n$, the quantity

$$
\frac{\partial \ell(y_n,s(x_n))}{\partial s(x_n)}
$$

measures how sensitive the loss is to the current score value $s(x_n)$. The negative of this quantity gives the direction in which we would like to move the score value in order to reduce the loss:

$$
-\frac{\partial \ell(y_n,s(x_n))}{\partial s(x_n)}.
$$

These negative gradient values are called **pseudo-residuals**:

$$
r_n=-\frac{\partial \ell(y_n,s(x_n))}{\partial s(x_n)}.
$$

They are the analogue of residuals for a general loss function. For squared error, they are exactly the usual residuals. For other losses, such as logistic loss, they are not ordinary residuals, but they still tell us the direction in which the current score should move at each training point.

In an idealized version of functional gradient descent, we would update the score values directly by

$$
s_m(x_n)=s_{m-1}(x_n)+\nu r_n^{(m)}.
$$

But this only defines updates at the training points. We need a function that can also make predictions at new values of $x$. 

**Gradient boosting** handles this by fitting a weak learner, usually a shallow regression tree, to the pseudo-residuals:

$$
\hat h_m(x_n) \approx r_n^{(m)}.
$$

Then the model is updated by adding this fitted function:

$$
\hat s_m(x)=\hat s_{m-1}(x)+\nu \hat h_m(x).
$$

Thus, gradient boosting can be viewed as gradient descent in function space, with the weak learner $\hat h_m$ serving as a smoothed, model-based approximation to the negative functional gradient.


## Generic Gradient Boosting Algorithm

Notice how our previous discussion worked for any differentiable loss $\ell$. This leads to the generic gradient boosting algorithm:

Input:

- data $(x_n,y_n)_{n=1}^N$
- differentiable loss $\ell(y,s)$
- weak learner class $\mathcal{H}$ (e.g. small trees)
- number of iterations $M$
- learning rate $\nu$

Initialize

$$
\hat s_0(x)=
\arg\min_c
\sum_{n=1}^N
\ell(y_n,c).
$$

For $m=1,\dots,M$:

1. Compute pseudo-residuals

$$
r_{nm}=
-\left[
\frac{\partial \ell(y_n,s(x_n))}
{\partial s(x_n)}
\right]_{s=\hat s_{m-1}}.
$$

2. Fit a weak learner to the pseudo-residuals

$$
\hat h_m \approx
\arg\min_{h \in \mathcal{H}}
\sum_{n=1}^N
(r_{nm}-h(x_n))^2.
$$

(Notice this is a regression task regardless of whether or not we are doing regression, classification with whatever loss.)

3. Boost

$$
\hat s_m(x)=
\hat s_{m-1}(x)
+
\nu \hat h_m(x).
$$

Return $\hat s_M$.


### Squared error as a special case

For squared error loss,

$$
\ell(y,s)=
\frac{1}{2}(y-s)^2.
$$

Then

$$
\frac{\partial \ell(y,s)}
{\partial s}=
s-y.
$$

So the negative gradient is

$$
-\frac{\partial \ell(y,s)}
{\partial s}=
y-s.
$$

At iteration $m$,

$$
r_{nm}=
y_n-\hat s_{m-1}(x_n).
$$

Thus ordinary residual boosting is exactly gradient boosting for squared error loss.


# Gradient boosting for binary classification

Now suppose

$$
y_n \in \{0,1\}.
$$

We use a real-valued score function $s(x)$ and convert it to a probability using the sigmoid:

$$
p(x)=
\sigma(s(x))=
\frac{1}{1+e^{-s(x)}}.
$$

The binary cross-entropy loss is

$$
\ell(y,s)=
-\left[
y\log \sigma(s)
+
(1-y)\log(1-\sigma(s))
\right].
$$

Equivalently,

$$
\ell(y,s)=
-y s+\log(1+e^s).
$$


Using

$$
p=\sigma(s),
$$

we have

$$
\frac{\partial \ell(y,s)}
{\partial s}=
p-y.
$$

Therefore the negative gradient is

$$
r=
y-p.
$$

At iteration $m$,

$$
p_{nm}=
\sigma(\hat s_{m-1}(x_n)),
$$

and

$$
r_{nm}=
y_n-p_{nm}.
$$

So binary logistic gradient boosting fits trees to the difference between the observed label and the current predicted probability.

The update is made on the **logit scale**:

$$
\hat s_m(x)=
\hat s_{m-1}(x)
+
\nu \hat h_m(x),
$$


In this case, the initial constant score is the minimizer of the empirical cross-entropy:

$$
\hat s_0=
\arg\min_c
\sum_{n=1}^N
\left[
-y_n c+\log(1+e^c)
\right].
$$

The solution is the empirical log-odds:

$$
\hat s_0=
\log\left(
\frac{\bar y}{1-\bar y}
\right),
$$

where

$$
\bar y=\frac{1}{N}\sum_{n=1}^N y_n.
$$

If the positive class is rare, this initial score is negative. If the positive class is common, it is positive.


## An Example

In [ ]:
X_cls, y_cls = make_moons(n_samples=600, noise=0.25, random_state=14211)

X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.35, random_state=93472
)

gb_clf = GradientBoostingClassifier(
    n_estimators=120, #this is M
    learning_rate=0.05, #this is nu
    max_depth=2,
    random_state=3928
)

gb_clf.fit(X_train, y_train)

In [ ]:
p_test = gb_clf.predict_proba(X_test)[:, 1]
y_pred = gb_clf.predict(X_test)

In [ ]:
p_test[:10]

In [ ]:
y_pred[:10]

In [ ]:
y_test[:10]

In [ ]:
print("Test accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
x1_min, x1_max = X_cls[:, 0].min() - 0.5, X_cls[:, 0].max() + 0.5
x2_min, x2_max = X_cls[:, 1].min() - 0.5, X_cls[:, 1].max() + 0.5

xx1, xx2 = np.meshgrid(
    np.linspace(x1_min, x1_max, 300),
    np.linspace(x2_min, x2_max, 300)
)

grid = np.c_[xx1.ravel(), xx2.ravel()]
prob_grid = gb_clf.predict_proba(grid)[:, 1].reshape(xx1.shape)

plt.figure(figsize=(7, 5))
plt.contourf(xx1, xx2, prob_grid, levels=20, alpha=0.7)
plt.contour(xx1, xx2, prob_grid, levels=[0.5], linewidths=2)
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolor="k", alpha=0.8)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Gradient boosted classifier")
plt.show()


# Gradient boosting for multiclass classification

Now suppose

$$
y_n \in \{1,\dots,K\}.
$$

As in multiclass logistic regression, we use one score per class:

$$
s_1(x),\dots,s_K(x).
$$

The softmax probabilities are

$$
p_k(x)=
\frac{\exp(s_k(x))}
{\sum_{\ell=1}^K \exp(s_\ell(x))}.
$$

Let $t_{nk}$ be the one-hot encoding of $y_n$:

$$
t_{nk}=
\mathbf{1}\{y_n=k\}.
$$

The multiclass cross-entropy loss is

$$
\ell(y_n,s(x_n))=
-\sum_{k=1}^K
t_{nk}\log p_k(x_n).
$$


For each class $k$,

$$
\frac{\partial \ell(y_n,s(x_n))}
{\partial s_k(x_n)}=
p_k(x_n)-t_{nk}.
$$

Therefore the negative gradient is

$$
r_{nkm}=t_{nk}-
p_{nk}^{(m-1)}.
$$

So at each boosting iteration we compute one pseudo-residual for every observation and every class.

A common procedure fits $K$ regression trees per boosting iteration:

For $m=1,\dots,M$:

1. Compute current probabilities

$$
p_{nk}^{(m-1)}=
\frac{\exp(\hat s_{k,m-1}(x_n))}
{\sum_{\ell=1}^K \exp(\hat s_{\ell,m-1}(x_n))}.
$$

2. For each class $k$, compute pseudo-residuals

$$
r_{nkm}=
t_{nk}-p_{nk}^{(m-1)}.
$$

3. Fit a regression tree $\hat h_{km}$ to $(x_n,r_{nkm})_{n=1}^N$.

4. Update each class score

$$
\hat s_{k,m}(x)=
\hat s_{k,m-1}(x)
+
\nu \hat h_{km}(x).
$$

The final prediction is

$$
\hat f(x)=
\arg\max_{1\le k\le K}
\hat s_{k,M}(x).
$$


# Regularization in boosting

Boosting can overfit, but the form of overfitting differs from a single deep tree.

The main regularization tools are:

### Number of trees

$$
M.
$$

More trees create a more complex additive function. Early stopping chooses $M$ using a validation set.

### Learning rate

$$
\nu.
$$

Smaller learning rates slow the stagewise fitting path. They usually require larger $M$.

### Tree depth

The depth of each tree controls the complexity of each update. Shallow trees produce simpler corrections.

Depth also controls interaction order in a rough sense:

- depth 1 trees capture main-effect style corrections
- depth 2 trees can capture simple interactions
- deeper trees can capture more complex interactions

### Minimum leaf size

Larger minimum leaf sizes prevent highly local updates.

### Subsampling

Some implementations fit each tree using a random subset of rows. This gives stochastic gradient boosting and can reduce variance.


# Practical notes

### Scaling

Boosted trees are usually not sensitive to monotone rescaling of individual features. A split such as

$$
x_j \le t
$$

has an equivalent split after changing units.

### Categorical variables

Basic CART implementations often require categorical variables to be encoded numerically. Different boosting libraries handle categorical features differently.

### Missing values

Basic sklearn trees require missing values to be handled before fitting. Some boosting implementations, including XGBoost-style methods, learn default directions for missing values.

### Extrapolation

Boosted trees inherit the extrapolation limitations of trees. They usually do not extrapolate linearly outside the observed feature range.


# Learning rate and early stopping on a real regression dataset

We now use the diabetes dataset to illustrate the interaction between learning rate and number of trees.


In [ ]:
diabetes = load_diabetes(as_frame=True)
X_diab = diabetes.data
y_diab = diabetes.target

X_train, X_test, y_train, y_test = train_test_split(
    X_diab, y_diab, test_size=0.35, random_state=18865
)

settings = [
    (1.0, 80),
    (0.3, 120),
    (0.1, 300),
    (0.03, 600),
]

results = []

for learning_rate, n_estimators in settings:
    model = GradientBoostingRegressor(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=2,
        random_state=0
    )
    model.fit(X_train, y_train)

    train_mse = []
    test_mse = []

    for pred_train, pred_test in zip(
        model.staged_predict(X_train),
        model.staged_predict(X_test)
    ):
        train_mse.append(mean_squared_error(y_train, pred_train))
        test_mse.append(mean_squared_error(y_test, pred_test))

    best_iteration = int(np.argmin(test_mse) + 1)
    results.append({
        "learning_rate": learning_rate,
        "n_estimators": n_estimators,
        "best_iteration": best_iteration,
        "best_test_mse": np.min(test_mse),
        "final_test_mse": test_mse[-1],
    })

    plt.figure(figsize=(7, 4))
    plt.plot(np.arange(1, len(train_mse) + 1), train_mse, label="train")
    plt.plot(np.arange(1, len(test_mse) + 1), test_mse, label="test")
    plt.axvline(best_iteration, linestyle="--", label="best test iteration")
    plt.xlabel("Boosting iteration")
    plt.ylabel("MSE")
    plt.title(f"learning_rate = {learning_rate}")
    plt.legend()
    plt.show()

display(pd.DataFrame(results))


The training loss decreases as more trees are added. The test loss may initially decrease and then flatten or increase.

This is why $M$ should usually be selected using validation error, cross-validation, or early stopping rather than training error.


# Step sizes, weak learners, and tree-specific updates

So far, we have described gradient boosting as repeatedly fitting a weak learner to the negative gradient and adding it to the current score function:

$$
\hat s_m(x)=\hat s_{m-1}(x)+\nu \hat h_m(x).
$$

This is the main idea, but there is a useful refinement. In ordinary gradient descent, there are really two separate questions at each iteration.

First, what direction should we move?

$$
d_m=-\nabla R(w_{m-1}).
$$

Second, how far should we move in that direction?

A simple gradient descent update uses a fixed learning rate:

$$
w_m=w_{m-1}+\nu d_m.
$$

But we could also choose a step size by minimizing the objective along that direction:

$$
\rho_m=\arg\min_\rho R(w_{m-1}+\rho d_m),
$$

and then update

$$
w_m=w_{m-1}+\rho_m d_m.
$$

The first problem is a **direction problem**. The second problem is a **step-size problem**. They are related, but they are not the same.

Gradient boosting has the same distinction. The negative functional gradient tells us the direction in which the current score values should move at the training points:

$$
r_{nm}=-\left.\frac{\partial \ell(y_n,s(x_n))}{\partial s(x_n)}\right|_{s=\hat s_{m-1}}.
$$

But we cannot usually add these values directly as a predictive function, because they are only defined at the training points. So we fit a weak learner to approximate this gradient direction:

$$
\hat h_m\approx\arg\min_{h\in\mathcal H}\sum_{n=1}^N (r_{nm}-h(x_n))^2.
$$

This fitting step is an approximation problem. It asks for a simple function, such as a shallow tree, whose predictions look like the negative gradient values.

Once we have chosen this weak learner, there is still a separate question: how large should the update be? A global line-search version would choose

$$
\rho_m=\arg\min_\rho \sum_{n=1}^N \ell(y_n,\hat s_{m-1}(x_n)+\rho \hat h_m(x_n)),
$$

and then update

$$
\hat s_m(x)=\hat s_{m-1}(x)+\nu\rho_m\hat h_m(x).
$$

Here, $\nu$ is the user-chosen shrinkage parameter, while $\rho_m$ is a loss-based step size chosen at iteration $m$.

This makes clear that there are two optimization problems happening:

1. fit $\hat h_m$ to approximate the negative gradient;
2. choose the step size for the update using the original loss.

When the weak learner is a tree, we can improve this further. A tree does not just give one direction. It gives terminal regions

$$
R_{1m},\ldots,R_{J_m m}.
$$

Within these regions, the tree update is piecewise constant. Instead of scaling the entire tree by one global number $\rho_m$, we can choose a different update in each leaf:

$$
\hat s_m(x)=\hat s_{m-1}(x)+\nu\sum_{j=1}^{J_m}\gamma_{jm}1(x\in R_{jm}).
$$

The leaf-specific update is chosen by minimizing the original loss inside that leaf:

$$
\gamma_{jm}=\arg\min_\gamma \sum_{x_n\in R_{jm}}\ell(y_n,\hat s_{m-1}(x_n)+\gamma).
$$

> This is the tree-specific version of step-size selection.

The tree fit to pseudo-residuals chooses the regions. The leaf updates choose how far to move in each region.

For squared-error loss, these two steps collapse: the optimal leaf update is just the mean residual in the leaf. For other losses, such as logistic loss, the mean pseudo-residual gives a first-order gradient direction, but the best finite additive update in a leaf may be different. This is why practical tree boosting often uses leaf-specific updates rather than a single global step size.


## Extensions and modern implementations

The core idea of gradient boosting is simple: compute pseudo-residuals, fit a weak learner to approximate them, and add the weak learner to the current score function. With trees, this is usually refined by using the fitted tree to define terminal regions and then choosing leaf-wise updates that reduce the original loss. Modern boosting methods build on this same idea but improve how the trees are fit, how the leaf values are computed, and how overfitting is controlled.

For example, the original `gbm` style algorithms follow the Friedman gradient boosting framework closely, with shrinkage, subsampling, and tree-based weak learners. XGBoost extends this by using second-order information, meaning both gradients and Hessians, to choose splits and compute regularized leaf values. LightGBM is designed for speed and scale; it uses efficient histogram-based split finding and grows trees leaf-wise, which can improve accuracy but also requires regularization. CatBoost adds machinery for handling categorical variables carefully, especially to reduce target leakage from naive encoding. Despite these implementation differences, the basic structure is the same: build an additive score function by sequentially adding trees that improve the current model.
